In [1]:
import pandas as pd

from sklearn.preprocessing import MinMaxScaler

from geoai.utils_ds.DataFrameOps import DataFrameOperations
df_ops = DataFrameOperations()


# READ DF

In [2]:
df = pd.read_csv("csv_files/dataset.csv")
print(df.head(5))
print(df.shape)

       BLUE      GREEN        RED        NIR     SWIR Landcover
0  835.3333   920.7500  1301.0000  1478.6666  1976.75   builtup
1  938.6667   966.3333  1153.0000  1376.0000  2025.20   builtup
2  871.0000   902.0000   946.0000  1218.0000  2025.20   builtup
3  985.0000  1050.0000  1114.6666  1573.0000  1974.75   builtup
4  940.2500   994.5000  1046.0000  1256.5000  1976.75   builtup
(2419, 6)


# SPLIT DATA

Data splitting involves dividing the dataset into separate subsets to train, validate, and test the model.
- Training set: Used to fit the model. It learns from this data.
- Validation set: Used to tune the parameters of the model. It helps in selecting the best model and configuring it properly.
- Test set: Used to assess the performance of the final model. It provides an unbiased evaluation.

In [3]:
X_train, X_val, X_test, y_train, y_val, y_test= df_ops.split_data(df, "Landcover")
X_train.to_csv("csv_files/X_train.csv", index=False)
X_val.to_csv("csv_files/X_val.csv", index=False)
X_test.to_csv("csv_files/X_test.csv", index=False)
print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

(1330, 5)
(605, 5)
(484, 5)


# USING DOMAIN KNOWLEDGE

Feature engineering is the process of creating or improving features. It is more of a dark art than a science. Features are often created based on common sense, domain knowledge, or prior experience. There are certain common techniques for feature creation; however, there is no guarantee that creating new features will improve results. 

For example, we can compute for NDVI, REI, and NDBI which can be derived from raw satellite data and serve as important features that can enhance the analysis and interpretation of remote sensing data.

$$ NDVI = \frac{NIR - RED}{NIR + RED} $$
$$ NDBI = \frac{SWIR - NIR}{SWIR + NIR} $$
$$ REI = \frac{NIR - BLUE}{NIR + BLUE * NIR} $$

In [4]:
X_train["NDVI"] = (X_train["NIR"] - X_train["RED"]) / (X_train["NIR"] + X_train["RED"])
X_train["NDBI"] = (X_train["SWIR"] - X_train["NIR"]) / (X_train["SWIR"] + X_train["NIR"])
X_train["REI"] = (X_train["NIR"] - X_train["BLUE"]) / (X_train["NIR"] + X_train["BLUE"] * X_train["NIR"])
X_train.fillna(0, inplace=True)
X_train.to_csv("csv_files/X_train_with_indices.csv", index=False)

X_val["NDVI"] = (X_val["NIR"] - X_val["RED"]) / (X_val["NIR"] + X_val["RED"])
X_val["NDBI"] = (X_val["SWIR"] - X_val["NIR"]) / (X_val["SWIR"] + X_val["NIR"])
X_val["REI"] = (X_val["NIR"] - X_val["BLUE"]) / (X_val["NIR"] + X_val["BLUE"] * X_val["NIR"])
X_val.fillna(0, inplace=True)
X_val.to_csv("csv_files/X_val_with_indices.csv", index=False)

X_test["NDVI"] = (X_test["NIR"] - X_test["RED"]) / (X_test["NIR"] + X_test["RED"])
X_test["NDBI"] = (X_test["SWIR"] - X_test["NIR"]) / (X_test["SWIR"] + X_test["NIR"])
X_test["REI"] = (X_test["NIR"] - X_test["BLUE"]) / (X_test["NIR"] + X_test["BLUE"] * X_test["NIR"])
X_test.fillna(0, inplace=True)
X_test.to_csv("csv_files/X_test_with_indices.csv", index=False)


# Polynomial transform

In [5]:
numerical_columns = X_train.select_dtypes(include=['float64']).columns.tolist()
print(numerical_columns)
X_train, X_val, X_test = df_ops.polynomial_transform(X_train, X_val, X_test, numerical_columns, 2)
X_train.head(1)

['BLUE', 'GREEN', 'RED', 'NIR', 'SWIR', 'NDVI', 'NDBI', 'REI']


,BLUE,GREEN,RED,NIR,SWIR,NDVI,NDBI,REI,BLUE^2,BLUE GREEN,...,SWIR^2,SWIR NDVI,SWIR NDBI,SWIR REI,NDVI^2,NDVI NDBI,NDVI REI,NDBI^2,NDBI REI,REI^2
344,1640.5,1671.6,1837.5,1918.3334,2692.8333,0.021522,0.167962,0.000088,2691240.25,2742259.8,...,7.251351e+06,57.955412,452.293152,0.23759,0.000463,0.003615,0.000002,0.028211,0.000015,7.784648e-09


# Binning/Discretization

 Separate feature values into several bins.

In [6]:
# Create binary NDVI
ndvi_binary_edges = [-float("inf"), 0.5, float("inf")]
ndvi_binary_labels = ["non_veg", "veg"]
X_train, X_val, X_test = df_ops.binarize_or_discretize(X_train, X_val, X_test, "NDVI", "NDVI_bin", ndvi_binary_edges, ndvi_binary_labels)
X_train.head(1)

,BLUE,GREEN,RED,NIR,SWIR,NDVI,NDBI,REI,BLUE^2,BLUE GREEN,...,SWIR NDVI,SWIR NDBI,SWIR REI,NDVI^2,NDVI NDBI,NDVI REI,NDBI^2,NDBI REI,REI^2,NDVI_bin
344,1640.5,1671.6,1837.5,1918.3334,2692.8333,0.021522,0.167962,0.000088,2691240.25,2742259.8,...,57.955412,452.293152,0.23759,0.000463,0.003615,0.000002,0.028211,0.000015,7.784648e-09,non_veg


In [7]:
# Create categorical NDVI
ndvi_category_edges = [-float("inf"), 0.2, 0.5, float("inf")]
ndvi_category_labels = ["low_veg", "medium_veg", "high_veg"]
X_train, X_val, X_test = df_ops.binarize_or_discretize(
    X_train, X_val, X_test, "NDVI", "NDVI_dis", ndvi_category_edges, ndvi_category_labels
)
X_train.head(1)

,BLUE,GREEN,RED,NIR,SWIR,NDVI,NDBI,REI,BLUE^2,BLUE GREEN,...,SWIR NDBI,SWIR REI,NDVI^2,NDVI NDBI,NDVI REI,NDBI^2,NDBI REI,REI^2,NDVI_bin,NDVI_dis
344,1640.5,1671.6,1837.5,1918.3334,2692.8333,0.021522,0.167962,0.000088,2691240.25,2742259.8,...,452.293152,0.23759,0.000463,0.003615,0.000002,0.028211,0.000015,7.784648e-09,non_veg,low_veg


In [8]:
X_train.columns 

Index(['BLUE', 'GREEN', 'RED', 'NIR', 'SWIR', 'NDVI', 'NDBI', 'REI', 'BLUE^2',
       'BLUE GREEN', 'BLUE RED', 'BLUE NIR', 'BLUE SWIR', 'BLUE NDVI',
       'BLUE NDBI', 'BLUE REI', 'GREEN^2', 'GREEN RED', 'GREEN NIR',
       'GREEN SWIR', 'GREEN NDVI', 'GREEN NDBI', 'GREEN REI', 'RED^2',
       'RED NIR', 'RED SWIR', 'RED NDVI', 'RED NDBI', 'RED REI', 'NIR^2',
       'NIR SWIR', 'NIR NDVI', 'NIR NDBI', 'NIR REI', 'SWIR^2', 'SWIR NDVI',
       'SWIR NDBI', 'SWIR REI', 'NDVI^2', 'NDVI NDBI', 'NDVI REI', 'NDBI^2',
       'NDBI REI', 'REI^2', 'NDVI_bin', 'NDVI_dis'],
      dtype='object')

# APPLY ONE HOT ENCODING

Process used to convert categorical data into a binary (0,1) vector representation where each category is represented by a unique vector in the space.

In [9]:
X_train, X_val, X_test = df_ops.make_one_hot_encoder(X_train, X_val, X_test, "NDVI_bin")
X_train.head(1)

,BLUE,GREEN,RED,NIR,SWIR,NDVI,NDBI,REI,BLUE^2,BLUE GREEN,...,SWIR REI,NDVI^2,NDVI NDBI,NDVI REI,NDBI^2,NDBI REI,REI^2,NDVI_dis,NDVI_bin_non_veg,NDVI_bin_veg
344,1640.5,1671.6,1837.5,1918.3334,2692.8333,0.021522,0.167962,0.000088,2691240.25,2742259.8,...,0.23759,0.000463,0.003615,0.000002,0.028211,0.000015,7.784648e-09,low_veg,1,0


# MAKE ORDINAL ENCODER

In [10]:
categories = [["low_veg", "medium_veg", "high_veg"]]
X_train, X_val, X_test = df_ops.make_ordinal_encoder(X_train, X_val, X_test, "NDVI_dis", categories)

In [11]:
X_train

,BLUE,GREEN,RED,NIR,SWIR,NDVI,NDBI,REI,BLUE^2,BLUE GREEN,...,SWIR REI,NDVI^2,NDVI NDBI,NDVI REI,NDBI^2,NDBI REI,REI^2,NDVI_bin_non_veg,NDVI_bin_veg,NDVI_dis_encoded
344,1640.5000,1671.6000,1837.50000,1918.3334,2692.8333,0.021522,0.167962,0.000088,2.691240e+06,2.742260e+06,...,0.237590,0.000463,0.003615,0.000002,0.028211,1.481938e-05,7.784648e-09,1,0,0
1024,952.0000,1048.6666,1328.00000,1422.0000,2032.0000,0.034182,0.176607,0.000347,9.063040e+05,9.983306e+05,...,0.704740,0.001168,0.006037,0.000012,0.031190,6.125095e-05,1.202848e-07,1,0,0
1512,1096.4000,1121.0000,1114.66660,1181.0000,1561.5000,0.028895,0.138742,0.000065,1.202093e+06,1.229064e+06,...,0.101929,0.000835,0.004009,0.000002,0.019249,9.056566e-06,4.260995e-09,1,0,0
2116,1816.6666,1870.0000,1831.50000,2164.5000,2117.3333,0.083333,-0.011016,0.000088,3.300278e+06,3.397167e+06,...,0.187193,0.006944,-0.000918,0.000007,0.000121,-9.738794e-07,7.816259e-09,1,0,0
2147,402.0000,549.0000,389.66666,2834.0000,1845.7500,0.758246,-0.211176,0.002129,1.616040e+05,2.206980e+05,...,3.930353,0.574937,-0.160123,0.001615,0.044595,-4.496793e-04,4.534374e-06,0,1,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2103,1862.6666,1963.3334,1951.33340,2271.0000,2059.0000,0.075709,-0.048961,0.000096,3.469527e+06,3.657036e+06,...,0.198649,0.005732,-0.003707,0.000007,0.002397,-4.723648e-06,9.308058e-09,1,0,0
1894,905.6667,1060.0000,1002.33330,2101.6000,2032.0000,0.354153,-0.016838,0.000628,8.202322e+05,9.600067e+05,...,1.275360,0.125424,-0.005963,0.000222,0.000284,-1.056793e-05,3.939295e-07,1,0,1
843,1096.6666,1195.0000,1395.00000,1691.5000,2556.7500,0.096064,0.203672,0.000320,1.202678e+06,1.310517e+06,...,0.819108,0.009228,0.019565,0.000031,0.041482,6.525059e-05,1.026374e-07,1,0,0
2204,386.5000,573.0000,364.25000,3120.5000,1864.0000,0.790946,-0.252081,0.002261,1.493822e+05,2.214645e+05,...,4.214524,0.625596,-0.199383,0.001788,0.063545,-5.699589e-04,5.112169e-06,0,1,2


# Scale numeric values

In [12]:
numerical_columns = X_train.select_dtypes(include=['float64']).columns.tolist()
scaler = MinMaxScaler()
X_train[numerical_columns] = scaler.fit_transform(X_train[numerical_columns])
X_val[numerical_columns] = scaler.transform(X_val[numerical_columns])
X_test[numerical_columns] = scaler.transform(X_test[numerical_columns])

X_train.to_csv("csv_files/X_train_preprocessed.csv", index=False)
X_val.to_csv("csv_files/X_val_preprocessed.csv", index=False)
X_test.to_csv("csv_files/X_test_preprocessed.csv", index=False)

# CONVERT LABEL STRING TO INTEGERS

In [13]:
y_train, y_val, y_test = df_ops.make_label_encoder(y_train, y_val, y_test)
y_train.to_csv("csv_files/y_train.csv", index=False)
y_val.to_csv("csv_files/y_val.csv", index=False)
y_test.to_csv("csv_files/y_test.csv", index=False)

{'builtup': 0, 'grass': 1, 'road': 2, 'trees': 3}


END